In [1]:
import pandas as pd
import json
import os
import sys

# Ensure src path is accessible
sys.path.append("../../src")
from genai_utils import get_groq_client

In [2]:
# Define fixed 7-category Theme Taxonomy
THEME_TAXONOMY = [
    "service_speed",
    "staff_behavior",
    "food_product_quality",
    "pricing_value",
    "cleanliness_ambiance",
    "order_accuracy_wait_time",
    "other_none"
]

In [3]:
# Load existing sentiment dataset and merchant priority dataset
sentiment_final_path = "../../data/samples/yelp_review_sentiment_final.csv"
merchant_priority_path = "../../data/samples/yelp_merchant_priority_final.csv"

df_sentiment = pd.read_csv(sentiment_final_path)
df_merchant_priority = pd.read_csv(merchant_priority_path)

print("--- Data Verification ---")
print(f"Sentiment dataset shape: {df_sentiment.shape}")
print(f"Merchant priority shape: {df_merchant_priority.shape}")
print("\nSentiment dataset columns:")
print(df_sentiment.columns.tolist())

--- Data Verification ---
Sentiment dataset shape: (2193, 4)
Merchant priority shape: (300, 40)

Sentiment dataset columns:
['review_id', 'sentiment_label', 'sentiment_score', 'sentiment_reason']


In [4]:
# Confirm review count and join keys
review_id_col = "review_id" if "review_id" in df_sentiment.columns else df_sentiment.columns[0]
print(f"\nJoin key identified: '{review_id_col}'")
print(f"Unique review IDs: {df_sentiment[review_id_col].nunique()}")


Join key identified: 'review_id'
Unique review IDs: 2193


In [5]:
# Display sample of input text to verify alignment
print("\nSample reviews ready for theme extraction:")
display_cols = [col for col in [review_id_col, "stars", "text", "sentiment"] if col in df_sentiment.columns]
print(df_sentiment[display_cols].head(3))


Sample reviews ready for theme extraction:
                review_id
0  t-8mpC0ryIc-cdnwoF8Hsw
1  rjnyXstBa7SUtlfrf-moyA
2  h46h5_2USejVTkfM3byl7g


In [6]:
# Load sample dataset containing original review text and stars
sample_path = "../../data/samples/yelp_sentiment_sample.csv"
df_sample = pd.read_csv(sample_path)

print(f"Sample dataset shape: {df_sample.shape}")
print(f"Sample dataset columns: {df_sample.columns.tolist()}")

Sample dataset shape: (2193, 11)
Sample dataset columns: ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date', 'period', 'merchant_status']


In [7]:
# Merge text and stars into df_sentiment
df_reviews = df_sentiment.merge(
    df_sample[['review_id', 'text', 'stars', 'business_id']], 
    on='review_id', 
    how='left'
)

print(f"\nMerged dataset shape: {df_reviews.shape}")
print(f"Null values in text column: {df_reviews['text'].isnull().sum()}")


Merged dataset shape: (2193, 7)
Null values in text column: 0


In [8]:
# Display sample of combined data
print("\nSample records ready for prompt construction:")
print(df_reviews[['review_id', 'stars', 'sentiment_label', 'text']].head(2).to_dict(orient='records'))


Sample records ready for prompt construction:
[{'review_id': 't-8mpC0ryIc-cdnwoF8Hsw', 'stars': 5, 'sentiment_label': 'positive', 'text': "A HIDDEN GEM!!!!!!!!! I had the Chelo Kabob Barg and Kashke Badenjan!  It was out of this world.  Delicious.  Very nice folks too.   Attached to a store.  Clean and accessible.   Enjoyed the music on the TV.  A wonderful experience!!!!! Can't wait to return!!!!!!"}, {'review_id': 'rjnyXstBa7SUtlfrf-moyA', 'stars': 1, 'sentiment_label': 'negative', 'text': "We had been to this restaurant before and it was better than average, but not last night when we took our guests there for dinner. I will summarize:\n1. We ordered appetizer (Bademjoon), it came very late (came with the dinner). It wasn't even tasty, too salty;\n2. The dinner came out 35 minutes after we ordered. Our Koobideh was hard as a rock and it had so much turmeric and fillers in it that tasted awful. Koobideh shouldn't have turmeric anyway but a little would be tolerable;\n3. Our Barg was

In [9]:
def build_theme_prompt(reviews_batch):
    """
    Constructs a prompt for extracting themes from a batch of reviews.
    reviews_batch: list of dicts with keys ['review_id', 'text', 'stars']
    """
    taxonomy_str = ", ".join(THEME_TAXONOMY)
    
    prompt = f"""You are a customer experience (CX) analyst evaluating restaurant reviews.

TAXONOMY OF PERMISSIBLE THEMES:
[{taxonomy_str}]

INSTRUCTIONS:
1. For each review provided below, select 1 to 2 themes from the PERMISSIBLE THEMES list that represent the primary topic(s) discussed in the review text.
2. If no category clearly applies, output ["other_none"].
3. Do NOT invent new theme names. Use ONLY the exact theme tags listed above.
4. Output MUST be a strictly valid JSON object with a single key "results".
5. The value of "results" must be a list of objects matching this exact format:

{{
  "results": [
    {{"review_id": "...", "themes": ["food_product_quality", "staff_behavior"]}},
    ...
  ]
}}

REVIEWS TO ANALYZE:
"""
    for item in reviews_batch:
        prompt += f"\nReview ID: {item['review_id']}\nStars: {item['stars']}\nText: {item['text']}\n---"

    return prompt

# Single test run on the first 2 sample reviews to verify prompt layout and JSON output
test_batch = df_reviews[['review_id', 'stars', 'text']].head(2).to_dict(orient='records')
client = get_groq_client()

print("Testing prompt payload generation:")
test_prompt = build_theme_prompt(test_batch)
print(test_prompt[:400] + "\n... [truncated] ...\n")

# Run single test call
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": test_prompt}],
    temperature=0.0
)

print("--- Test LLM Response Output ---")
print(response.choices[0].message.content)

Testing prompt payload generation:
You are a customer experience (CX) analyst evaluating restaurant reviews.

TAXONOMY OF PERMISSIBLE THEMES:
[service_speed, staff_behavior, food_product_quality, pricing_value, cleanliness_ambiance, order_accuracy_wait_time, other_none]

INSTRUCTIONS:
1. For each review provided below, select 1 to 2 themes from the PERMISSIBLE THEMES list that represent the primary topic(s) discussed in the review 
... [truncated] ...



2026-09-02 17:53:00,269 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


--- Test LLM Response Output ---
```json
{
  "results": [
    {
      "review_id": "t-8mpC0ryIc-cdnwoF8Hsw",
      "themes": ["food_product_quality", "staff_behavior"]
    },
    {
      "review_id": "rjnyXstBa7SUtlfrf-moyA",
      "themes": ["service_speed", "food_product_quality"]
    }
  ]
}
```


In [10]:
import time
import re

checkpoint_path = "../../data/raw/yelp/yelp_review_themes_checkpoint.csv"

# Load checkpoint if it exists
if os.path.exists(checkpoint_path):
    df_checkpoint = pd.read_csv(checkpoint_path)

    # Treat reviews with invalid "none" themes as unprocessed
    valid_checkpoint = df_checkpoint[
        df_checkpoint["themes"].fillna("").apply(
            lambda x: all(
                theme.strip() in THEME_TAXONOMY
                for theme in str(x).split(",")
            )
        )
    ].copy()

    processed_ids = set(valid_checkpoint["review_id"].tolist())
    print(f"Found existing checkpoint with {len(df_checkpoint)} saved reviews.")
    print(f"Valid processed reviews: {len(processed_ids)}")
    print(f"Reviews requiring re-processing: {len(df_checkpoint) - len(processed_ids)}")

else:
    df_checkpoint = pd.DataFrame(columns=["review_id", "themes"])
    processed_ids = set()
    print("No checkpoint found. Starting fresh run.")

# Filter remaining un-processed reviews
df_remaining = df_reviews[~df_reviews["review_id"].isin(processed_ids)].copy()
print(f"Total reviews remaining to process: {len(df_remaining)}")

BATCH_SIZE = 10
records_to_process = df_remaining[["review_id", "stars", "text"]].to_dict(orient="records")

def clean_json_response(raw_text):
    """Extract JSON object from an LLM response, including markdown-wrapped JSON."""

    raw_text = raw_text.strip()

    # Remove markdown code fences if present
    raw_text = re.sub(
        r"^```(?:json)?\s*",
        "",
        raw_text,
        flags=re.IGNORECASE
    )

    raw_text = re.sub(
        r"\s*```$",
        "",
        raw_text
    )

    # Extract the complete JSON object
    start = raw_text.find("{")
    end = raw_text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found in LLM response.")

    return raw_text[start:end + 1]

# Run batch processing loop
for i in range(0, len(records_to_process), BATCH_SIZE):
    batch = records_to_process[i:i + BATCH_SIZE]
    batch_ids = [r["review_id"] for r in batch]
    prompt = build_theme_prompt(batch)
    success = False

    # Retry malformed/invalid responses up to 3 times
    for attempt in range(1, 4):
        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=[
                    {"role": "user", "content": prompt}
                ],
                temperature=0.0,
                response_format={"type": "json_object"}
            )

            raw_content = response.choices[0].message.content
            print("\n--- RAW LLM RESPONSE ---")
            print(raw_content)
            print("--- END RAW RESPONSE ---\n")

            cleaned_content = clean_json_response(raw_content)
            parsed_response = json.loads(cleaned_content)

            if "results" not in parsed_response:
                raise ValueError("LLM response is missing the 'results' key.")

            parsed_batch = parsed_response["results"]

            # Validate response before saving
            if not isinstance(parsed_batch, list):
                raise ValueError("LLM response is not a JSON list.")

            if len(parsed_batch) != len(batch):
                raise ValueError(
                    f"Expected {len(batch)} results, "
                    f"received {len(parsed_batch)}."
                )

            returned_ids = [
                str(item["review_id"])
                for item in parsed_batch
            ]

            # Check that all expected IDs were returned
            if set(returned_ids) != set(batch_ids):
                raise ValueError(
                    "Returned review IDs do not match the batch IDs."
                )

            batch_results = []

            for item in parsed_batch:
                themes = item.get(
                    "themes",
                    ["other_none"]
                )

                # Validate themes
                if not isinstance(themes, list):
                    raise ValueError(
                        f"Themes must be a list for review "
                        f"{item['review_id']}"
                    )

                if not 1 <= len(themes) <= 2:
                    raise ValueError(
                        f"Review {item['review_id']} has "
                        f"{len(themes)} themes; expected 1-2."
                    )

                invalid_themes = [
                    theme
                    for theme in themes
                    if theme not in THEME_TAXONOMY
                ]

                if invalid_themes:
                    raise ValueError(
                        f"Invalid theme(s): {invalid_themes}"
                    )

                # Format themes as comma-separated string
                themes_str = ",".join(themes)

                batch_results.append({
                    "review_id": item["review_id"],
                    "themes": themes_str
                })

            df_batch = pd.DataFrame(batch_results)

            # Save only after successful validation
            df_checkpoint = pd.concat(
                [df_checkpoint, df_batch],
                ignore_index=True
            )

            # Replace any previous result for the same review_id
            df_checkpoint = df_checkpoint.drop_duplicates(
                subset=["review_id"],
                keep="last"
            )

            df_checkpoint.to_csv(
                checkpoint_path,
                index=False
            )

            print(
                f"Processed batch {i // BATCH_SIZE + 1} / "
                f"{(len(records_to_process) - 1) // BATCH_SIZE + 1} "
                f"({len(df_checkpoint)} total saved)"
            )

            success = True
            break

        except Exception as e:
            print(
                f"Batch {i // BATCH_SIZE + 1} "
                f"attempt {attempt}/3 failed: {e}"
            )

            if attempt < 3:
                print("Retrying in 3 seconds...")
                time.sleep(3)

                # Add explicit instruction for retry
                prompt = build_theme_prompt(batch)
                prompt += """
                    IMPORTANT:

                    Return ONLY a strictly valid JSON object.

                    The JSON object must contain a single key named "results".
                    The value of "results" must be the list of review results.

                    Do not use markdown code fences.
                    Do not include explanations or additional text.
                    Make sure every review_id appears exactly once.

                    """

            else:

                print("Batch failed after 3 attempts. "
                      "Previous progress is safely saved to checkpoint."
                     )

    # Stop if this batch could not be processed
    if not success:
        print(
            "Processing stopped. "
            "Re-run the cell later to resume."
        )
        break

    # Brief pause between successful batches
    time.sleep(1)


print(
    f"\nProcessing step complete/paused. "
    f"Current checkpoint count: {len(df_checkpoint)}"
)

Found existing checkpoint with 2194 saved reviews.
Valid processed reviews: 2189
Reviews requiring re-processing: 5
Total reviews remaining to process: 5


2026-09-02 17:53:06,515 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



--- RAW LLM RESPONSE ---
{"results":[{"review_id":"ixuFWi5Sxr_FvQx0r8G_sg","themes":["other_none"]},{"review_id":"3TKXvQwW8bdqID73Tk_22Q","themes":["other_none"]},{"review_id":"toYOIBEG6bfYaIfUqmoL3A","themes":["food_product_quality"]},{"review_id":"rqYxUbOIOx_vR8KU38E1Aw","themes":["staff_behavior"]},{"review_id":"Cnh9aQEKWDkaAipRMDx5ww","themes":["other_none"]}]}
--- END RAW RESPONSE ---

Processed batch 1 / 1 (2194 total saved)

Processing step complete/paused. Current checkpoint count: 2194


In [11]:
import os
import pandas as pd

checkpoint_path = "../../data/raw/yelp/yelp_review_themes_checkpoint.csv"
final_output_path = "../../data/samples/yelp_review_themes_final.csv"

# Load checkpoint data
df_checkpoint = pd.read_csv(checkpoint_path)

In [12]:
# Final cleanup and validation
# Remove the known stray review that is not present in the source sample
stray_review_id = "hUMXVLRvj_WcqDc8q_GLQ"

df_checkpoint = df_checkpoint[
    df_checkpoint["review_id"] != stray_review_id
].copy()

# Remove any duplicate review IDs, keeping the latest result
df_final = df_checkpoint.drop_duplicates(
    subset=["review_id"],
    keep="last"
).copy()

# Check for invalid "none" themes
invalid_none = df_final[
    df_final["themes"].fillna("").str.split(",").apply(
        lambda themes: "none" in [theme.strip() for theme in themes]
    )
]

print(f"Invalid 'none' records remaining: {len(invalid_none)}")

if len(invalid_none) > 0:
    print(invalid_none.to_string(index=False))
    raise ValueError("Invalid 'none' themes remain in final output.")

# Keep only the required columns
df_final = df_final[["review_id", "themes"]]

# Save cleaned final output
df_final.to_csv(
    final_output_path,
    index=False
)

print(f"Final output saved to: {final_output_path}")
print(f"Total unique reviews saved: {len(df_final)}")

Invalid 'none' records remaining: 0
Final output saved to: ../../data/samples/yelp_review_themes_final.csv
Total unique reviews saved: 2193


In [13]:
import glob
import os
import pandas as pd

# 1. Load datasets
df_themes = pd.read_csv("../../data/samples/yelp_review_themes_final.csv")
df_merchant = pd.read_csv("../../data/samples/yelp_merchant_priority_final.csv")

# 2. Load ALL review parquet files to get complete review_id -> business_id & text mapping
parquet_files = sorted(
    glob.glob("../../data/raw/yelp/yelp_review_sample_part_*.parquet")
)
df_reviews_all = pd.concat(
    [
        pd.read_parquet(f, columns=["review_id", "business_id", "stars", "text"])
        for f in parquet_files
    ],
    ignore_index=True,
)

# 3. Complete Spot-Check on 5 reviews with full text visible
df_merged_check = df_themes.merge(df_reviews_all, on="review_id", how="inner")

print("--- FULL TEXT SPOT-CHECK (FIRST 5 REVIEWS) ---")
for idx, row in df_merged_check.head(5).iterrows():
    print(f"[{row['stars']}★] Review ID: {row['review_id']}")
    print(f"Themes: {row['themes']}")
    print(f"Text: {row['text'][:150]}...\n" + "-" * 60)

# 4. Map themes to business_id and explode individual themes
df_exploded = df_merged_check.assign(
    theme=df_merged_check["themes"].str.split(",")
).explode("theme")
df_exploded["theme"] = df_exploded["theme"].str.strip()

# 5. Group by business_id and theme to get count matrix
theme_counts = (
    df_exploded.groupby(["business_id", "theme"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# 6. Merge with Merchant Priority metadata
df_merchant_summary = df_merchant[
    ["business_id", "name", "merchant_status", "priority_group"]
].merge(theme_counts, on="business_id", how="left")
df_merchant_summary.fillna(0, inplace=True)

# 7. Save merchant theme summary CSV
output_path = "../../outputs/tables/merchant_theme_summary.csv"
df_merchant_summary.to_csv(output_path, index=False)
print(f"Saved merchant theme summary to: {output_path}")

# 8. Compute Top CX Drivers by Merchant Health Tier
df_status_themes = (
    df_merchant_summary.merge(
        df_exploded[["business_id", "theme"]], on="business_id"
    )
    .groupby(["merchant_status", "theme"])
    .size()
    .unstack(fill_value=0)
)

print("\n--- THEME CONCENTRATION BY MERCHANT HEALTH TIER ---")
print(df_status_themes)

--- FULL TEXT SPOT-CHECK (FIRST 5 REVIEWS) ---
[5★] Review ID: t-8mpC0ryIc-cdnwoF8Hsw
Themes: food_product_quality,staff_behavior
Text: A HIDDEN GEM!!!!!!!!! I had the Chelo Kabob Barg and Kashke Badenjan!  It was out of this world.  Delicious.  Very nice folks too.   Attached to a sto...
------------------------------------------------------------
[1★] Review ID: rjnyXstBa7SUtlfrf-moyA
Themes: service_speed,staff_behavior
Text: We had been to this restaurant before and it was better than average, but not last night when we took our guests there for dinner. I will summarize:
1...
------------------------------------------------------------
[5★] Review ID: h46h5_2USejVTkfM3byl7g
Themes: food_product_quality
Text: I am half-Iranian and absolutely LOVE Persian cuisine. When I lived in South Florida, I had a number of good restaurants from which to choose... so up...
------------------------------------------------------------
[4★] Review ID: jQHWfUlgObQRh40PUDYU_w
Themes: food_product_qua

In [14]:
import pandas as pd

# Load merchant theme summary table
df_merchant_summary = pd.read_csv("../../outputs/tables/merchant_theme_summary.csv")

# Filter for Declining merchants and sum theme frequencies
theme_cols = [
    c
    for c in df_merchant_summary.columns
    if c not in ["business_id", "name", "merchant_status", "priority_group"]
]
declining_themes = (
    df_merchant_summary[
        df_merchant_summary["merchant_status"] == "Declining"
    ][theme_cols]
    .sum()
    .sort_values(ascending=False)
)

# Build Top 3 CX Drivers Table
total_declining_mentions = declining_themes.sum()
top_3_df = pd.DataFrame(
    {
        "Theme Driver": declining_themes.index[:3],
        "Mentions": declining_themes.values[:3],
        "Share of Mentions (%)": (
            declining_themes.values[:3] / total_declining_mentions * 100
        ).round(2),
    }
)

print("--- TOP 3 CX DRIVERS OF DECLINING MERCHANTS ---")
print(top_3_df.to_string(index=False))

--- TOP 3 CX DRIVERS OF DECLINING MERCHANTS ---
        Theme Driver  Mentions  Share of Mentions (%)
food_product_quality       597                  45.06
      staff_behavior       317                  23.92
       pricing_value       138                  10.42


### Add Top CX Theme to Merchant Priority Table

For each sentiment-covered merchant, identify the one or two most frequently occurring extracted CX themes and merge them into the final merchant priority table.

In [15]:
import pandas as pd

# Load merchant theme summary and final priority table
theme_summary_path = "../../outputs/tables/merchant_theme_summary.csv"
priority_path = "../../data/samples/yelp_merchant_priority_final.csv"

df_theme_summary = pd.read_csv(theme_summary_path)
df_priority = pd.read_csv(priority_path)

print(f"Theme summary rows: {len(df_theme_summary)}")
print(f"Priority table rows: {len(df_priority)}")

# Define the seven permissible CX themes
theme_columns = [
    "service_speed",
    "staff_behavior",
    "food_product_quality",
    "pricing_value",
    "cleanliness_ambiance",
    "order_accuracy_wait_time",
    "other_none"
]

# Identify the top 1-2 themes for each merchant
def get_top_cx_themes(row):
    theme_counts = row[theme_columns].astype(float)

    # Keep only themes that were actually mentioned
    theme_counts = theme_counts[theme_counts > 0]

    if theme_counts.empty:
        return "other_none"

    # Sort from most frequent to least frequent
    theme_counts = theme_counts.sort_values(ascending=False)

    # Keep the top 2 themes
    top_themes = theme_counts.index[:2].tolist()

    return ",".join(top_themes)

df_theme_summary["top_cx_theme"] = (
    df_theme_summary.apply(get_top_cx_themes, axis=1)
)

# Keep only the columns needed for the merge
merchant_top_themes = df_theme_summary[
    ["business_id", "top_cx_theme"]
].copy()

# Validate that business_id is unique before merging
duplicate_ids = merchant_top_themes[
    merchant_top_themes["business_id"].duplicated()
]

if len(duplicate_ids) > 0:
    raise ValueError(
        f"Found {len(duplicate_ids)} duplicate business_id values "
        "in merchant theme summary."
    )

# Merge top CX theme into priority table
df_priority = df_priority.drop(
    columns=["top_cx_theme"],
    errors="ignore"
)

df_priority = df_priority.merge(
    merchant_top_themes,
    on="business_id",
    how="left",
    validate="one_to_one"
)

# Validate merge coverage
missing_themes = df_priority["top_cx_theme"].isna().sum()

print(f"Priority table rows after merge: {len(df_priority)}")
print(f"Merchants with top CX theme: {len(df_priority) - missing_themes}")
print(f"Merchants missing top CX theme: {missing_themes}")

if len(df_priority) != 300:
    raise ValueError(
        f"Expected 300 merchants after merge, "
        f"found {len(df_priority)}."
    )

if missing_themes > 0:
    raise ValueError(
        f"{missing_themes} merchants were not matched "
        "to a top CX theme."
    )

# Save updated priority table
df_priority.to_csv(
    priority_path,
    index=False
)

print(
    f"\nSaved updated priority table to:\n{priority_path}"
)

print("\nNew column:")
print(df_priority["top_cx_theme"].head(10))

Theme summary rows: 300
Priority table rows: 300
Priority table rows after merge: 300
Merchants with top CX theme: 300
Merchants missing top CX theme: 0

Saved updated priority table to:
../../data/samples/yelp_merchant_priority_final.csv

New column:
0          food_product_quality,staff_behavior
1          food_product_quality,staff_behavior
2           food_product_quality,pricing_value
3          staff_behavior,food_product_quality
4      staff_behavior,order_accuracy_wait_time
5    food_product_quality,cleanliness_ambiance
6           food_product_quality,service_speed
7          food_product_quality,staff_behavior
8          food_product_quality,staff_behavior
9          food_product_quality,staff_behavior
Name: top_cx_theme, dtype: str
